# nb35 - Pre-declared final pair: D4-symmetry TTA and a high-E specialist

Record: 7-model ensemble 0.0444, per-bin 0.0668/0.0501/0.0374/0.0379/0.0368/0.0373 (nb34). Two principled, pre-declared additions - tested once, reported as they land, no iteration (guarding against test-split tuning):

1. **Test-time augmentation over the dihedral group.** The cell grid has an (approximate) 8-fold symmetry: flips and 90-degree rotations of (di, dj) produce equally-valid views of the same shower. Averaging the frozen models' predictions over all 8 views is a symmetry prior, not a fit - zero new parameters.
2. **High-E specialist.** The remaining gap lives at E>17 GeV. A model trained only on events with ensemble-PREDICTED energy > 15 GeV (prediction-based gating - no truth leakage) can specialize its capacity there, the ML analogue of range-based calibration used by every experiment. Two seeds, gated at inference by the same predicted-energy cut.

In [1]:
import os, sys, copy, time, pathlib
import numpy as np, pandas as pd, uproot, awkward as ak
import torch, torch.nn as nn
REPO = pathlib.Path(os.environ['REPO_DIR']) if os.environ.get('REPO_DIR') else (
    pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd())
sys.path.insert(0, str(REPO / 'scripts'))
from run_experiments import split, resolution, PITCH, EPS
MB = sorted((REPO / 'data' / 'minimum_bias').glob('*.root'))
CLEANF = sorted((REPO / 'data' / 'full').glob('matched_*.root'))
OUT = REPO / 'reports' / 'predictions'
CKPT = REPO / '.scratch' / 'ckpt'
DEVICE = os.environ.get('NB35_DEVICE') or ('cuda' if torch.cuda.is_available() else 'cpu')
MODE = os.environ.get('NB35_MODE', 'full')
if MODE == 'smoke': MB, CLEANF = MB[:8], CLEANF[:4]
THRESH = 2.49
W = 4; L = (2*W+1)**2
print('device', DEVICE, '| mode', MODE, '|', len(MB), 'mb files')

device cuda | mode full | 94 mb files


In [2]:
TK = ['cell_x','cell_y','energy','cell_energies_front','cell_energies_back',
      'cell_times_front','cell_times_back','imodx','jmody']
AUX = ['sig_flux_prod_vertex_z','sig_flux_eTot']
def event_geom(cc):
    x, yy, e = cc['cell_x'], cc['cell_y'], cc['energy']
    ix, iy = cc['imodx'], cc['jmody']
    seed = int(np.argmax(e))
    pts = np.stack([x, yy], 1)
    pitch = np.full(len(x), np.nan)
    for key in {(int(p), int(q)) for p, q in zip(ix, iy)}:
        sel = (ix == key[0]) & (iy == key[1]); p = pts[sel]
        if len(p) >= 2:
            d = np.sqrt(((p[:, None, :] - p[None, :, :]) ** 2).sum(-1)); d[d == 0] = np.inf
            pitch[sel] = np.median(np.min(d, axis=1))
    fill = np.nanmedian(pitch) if np.isfinite(pitch).any() else 120.0
    pitch[~np.isfinite(pitch)] = fill
    ps = pitch[seed]
    ei = (x - x[seed]) / ps; ej = (yy - yy[seed]) / ps
    di = np.round(ei).astype(int); dj = np.round(ej).astype(int)
    ok = (np.abs(ei - di) < 0.15) & (np.abs(ej - dj) < 0.15)
    return seed, ps, di, dj, ok
def build_grid(files, label):
    EV = []
    for path in files:
        with uproot.open(path) as f:
            a = f['clusters_matched'].arrays(TK + AUX, library='ak')
        vz = ak.to_numpy(a['sig_flux_prod_vertex_z']).astype(float)
        et_all = ak.to_numpy(a['sig_flux_eTot']).astype(float)
        for i in np.flatnonzero((vz < 100.0) & (et_all >= 1.0) & (et_all <= 100.0)):
            cc = {k: np.asarray(ak.to_numpy(a[k][i])).astype(float) for k in TK}
            e = cc['energy']
            if len(e) < 3: continue
            seed, ps, di, dj, ok = event_geom(cc)
            if ok.mean() < 0.5 or not ok[seed]: continue
            tf = cc['cell_times_front']; tb = cc['cell_times_back']
            tf = np.where(np.isfinite(tf) & (tf != 0) & (np.abs(tf) < 1e4), tf, np.nan)
            tb = np.where(np.isfinite(tb) & (tb != 0) & (np.abs(tb) < 1e4), tb, np.nan)
            EV.append(dict(di=di[ok].astype(np.int16), dj=dj[ok].astype(np.int16),
                           e=e[ok].astype(np.float32),
                           fr=cc['cell_energies_front'][ok].astype(np.float32),
                           bk=cc['cell_energies_back'][ok].astype(np.float32),
                           tf=tf[ok].astype(np.float32), tb=tb[ok].astype(np.float32),
                           ps=float(ps), reg=int(np.argmin(np.abs(PITCH - ps))),
                           Etrue=float(et_all[i])))
    print(f'{label}: {len(EV)} events')
    return EV
ME = build_grid(MB, 'minbias')
CE = build_grid(CLEANF, 'clean')

minbias: 72554 events


clean: 30303 events


In [3]:
def make_windows(EVS):
    rows = []; keep = []
    for i, ev in enumerate(EVS):
        m = (np.maximum(np.abs(ev['di']), np.abs(ev['dj'])) <= W) & (ev['e'] >= THRESH)
        if m.sum() < 1: continue
        di, dj, e, fr, bk, tf, tb = (v[m] for v in (ev['di'], ev['dj'], ev['e'], ev['fr'], ev['bk'], ev['tf'], ev['tb']))
        t0f = np.nanmedian(tf) if np.isfinite(tf).any() else 0.0
        t0b = np.nanmedian(tb) if np.isfinite(tb).any() else 0.0
        tfc = np.where(np.isfinite(tf), tf - t0f, 0.0); htf = np.isfinite(tf).astype(np.float32)
        tbc = np.where(np.isfinite(tb), tb - t0b, 0.0); htb = np.isfinite(tb).astype(np.float32)
        rdr = np.hypot(di, dj)
        cont = np.stack([np.log1p(np.clip(e, 0, None)), np.log1p(np.clip(fr, 0, None)),
                         np.log1p(np.clip(bk, 0, None)), di.astype(np.float32), dj.astype(np.float32),
                         rdr, np.full(len(e), np.log(ev['ps'])), np.clip(tfc, -5, 5), np.clip(tbc, -5, 5)], 1)
        oh = np.zeros((len(e), len(PITCH)), np.float32); oh[:, ev['reg']] = 1.0
        tok = np.concatenate([cont, htf[:, None], htb[:, None], oh], 1).astype(np.float32)
        rows.append((tok, float(e.sum()), float(e.max()), ev['Etrue'])); keep.append(i)
    return rows, np.array(keep)
NC = 9
rows_mb, keep = make_windows(ME)
remap = -np.ones(len(ME), int); remap[keep] = np.arange(len(keep))
a_, b_, t_ = split(len(ME))
ktr = remap[a_][remap[a_] >= 0]; kva = remap[b_][remap[b_] >= 0]; kte = remap[t_][remap[t_] >= 0]
rows_cl, _ = make_windows(CE)
n_mb = len(rows_mb); rows = rows_mb + rows_cl
ctr = np.arange(n_mb, len(rows))
N = len(rows); IN_DIM = rows[0][0].shape[1]
y = np.array([np.log(max(r[3], 1e-3)) for r in rows], np.float32)
Et = np.array([r[3] for r in rows], np.float32)
sumE = np.array([r[1] for r in rows], np.float32)
X = np.zeros((N, L, IN_DIM), np.float32); M = np.zeros((N, L), np.bool_)
G5 = np.zeros((N, 5), np.float32); Eraw = np.zeros((N, L), np.float32)
for i, (tok, se, sde, et) in enumerate(rows):
    n = tok.shape[0]; X[i, :n] = tok; M[i, :n] = True
    e = np.expm1(tok[:, 0]); Eraw[i, :n] = e
    lat = float(np.sqrt((e * tok[:, 5] ** 2).sum() / (e.sum() + EPS)))
    fbr = float(np.expm1(tok[:, 1]).sum() / (np.expm1(tok[:, 2]).sum() + EPS))
    G5[i] = [np.log1p(se), np.log1p(sde), np.log(n), fbr, lat]
la0, lb0 = np.polyfit(np.log1p(0.5 * sumE[ktr]), y[ktr], 1)
G5 = (G5 - G5[ktr].mean(0)) / (G5[ktr].std(0) + EPS)
cont = X[ktr][:, :, :NC].reshape(-1, NC)[M[ktr].reshape(-1)]
mean = cont.mean(0); std = cont.std(0) + EPS
X[:, :, :NC] = (X[:, :, :NC] - mean) / std; X[~M] = 0.0
Xc = torch.from_numpy(X).to(DEVICE); Mc = torch.from_numpy(M).to(DEVICE)
G5c = torch.from_numpy(G5).to(DEVICE); G6c = torch.cat([G5c, torch.zeros(N, 1, device=DEVICE)], 1)
Ec = torch.from_numpy(Eraw).to(DEVICE); Yc = torch.from_numpy(y).unsqueeze(1).to(DEVICE)
print('N', N, '(mb', n_mb, '+ clean', len(ctr), ') tr/va/te', len(ktr), len(kva), len(kte))

N 102857 (mb 72554 + clean 30303 ) tr/va/te 50787 10883 10884


In [4]:
CFG = dict(d=128, nhead=4, layers=3, dropout=0.1, lr=3e-4, wd=1e-4, batch=96, huber_delta=0.1)
class SubNet(nn.Module):
    def __init__(self, in_dim, ng, la0, lb0):
        super().__init__()
        d = CFG['d']
        self.embed = nn.Linear(in_dim, d)
        layer = nn.TransformerEncoderLayer(d, CFG['nhead'], dim_feedforward=4*d,
                                           dropout=CFG['dropout'], batch_first=True)
        self.enc = nn.TransformerEncoder(layer, CFG['layers'], enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d)
        self.head = nn.Sequential(nn.Linear(d + ng, d), nn.ReLU(), nn.Dropout(CFG['dropout']), nn.Linear(d, 1))
        self.fhead = nn.Sequential(nn.Linear(d, d // 2), nn.ReLU(), nn.Linear(d // 2, 1))
        self.la = nn.Parameter(torch.tensor(float(la0))); self.lb = nn.Parameter(torch.tensor(float(lb0)))
    def forward(self, x, m, g, ecell):
        h = self.enc(self.embed(x), src_key_padding_mask=~m)
        fl = self.fhead(h).squeeze(-1)
        w = torch.sigmoid(fl) * m.float()
        base = self.la * torch.log1p((w * ecell).sum(1, keepdim=True)) + self.lb
        wm = m.unsqueeze(-1).float()
        p = self.norm((h * wm).sum(1) / wm.sum(1).clamp(min=1))
        return base + self.head(torch.cat([p, g], 1))
MODELS = []
for name, ng in [('nb32_W4_s0', 5), ('nb32_W4_s1', 5), ('nb32_W4_s2', 5),
                 ('nb34_pure_s3', 6), ('nb34_pure_s4', 6), ('nb34_cleanaux_s0', 6), ('nb34_cleanaux_s1', 6)]:
    ck = CKPT / f'{name}.pt'
    if not ck.exists():
        print('missing', name); continue
    mdl = SubNet(IN_DIM, ng, la0, lb0).to(DEVICE)
    mdl.load_state_dict(torch.load(ck, map_location=DEVICE)['bstate']); mdl.eval()
    MODELS.append((name, ng, mdl))
print('loaded', [m[0] for m in MODELS])

loaded ['nb32_W4_s0', 'nb32_W4_s1', 'nb32_W4_s2', 'nb34_pure_s3', 'nb34_pure_s4', 'nb34_cleanaux_s0', 'nb34_cleanaux_s1']


## Part 1 - D4 test-time augmentation
The 8 dihedral transforms act on the (di, dj) token columns only (rdr and all globals are invariant). Predictions are averaged in log space per model, then across models. Report raw ensemble vs TTA ensemble on the real test split.

In [5]:
DI, DJ = 3, 4
def tta_views(xb, mb):
    di = xb[:, :, DI] * std[DI] + mean[DI]; dj = xb[:, :, DJ] * std[DJ] + mean[DJ]
    for swap in (False, True):
        for s1 in (1.0, -1.0):
            for s2 in (1.0, -1.0):
                a = (dj if swap else di) * s1; b = (di if swap else dj) * s2
                xv = xb.clone()
                xv[:, :, DI] = torch.where(mb, (a - mean[DI]) / std[DI], torch.zeros_like(a))
                xv[:, :, DJ] = torch.where(mb, (b - mean[DJ]) / std[DJ], torch.zeros_like(b))
                yield xv
meanT = torch.tensor(mean, device=DEVICE); stdT = torch.tensor(std, device=DEVICE)
mean = meanT; std = stdT
def infer(idx, tta):
    per_model = []
    with torch.no_grad():
        for name, ng, mdl in MODELS:
            g = G5c if ng == 5 else G6c
            out = []
            for j in range(0, len(idx), 256):
                b = torch.from_numpy(np.asarray(idx[j:j+256])).to(DEVICE)
                xb, mb2 = Xc[b], Mc[b]
                if tta:
                    preds = [mdl(xv, mb2, g[b], Ec[b]) for xv in tta_views(xb, mb2)]
                    out.append(torch.stack(preds).mean(0).cpu().numpy().ravel())
                else:
                    out.append(mdl(xb, mb2, g[b], Ec[b]).cpu().numpy().ravel())
            per_model.append(np.concatenate(out))
    return np.stack(per_model)
def calib_eval(raw_va, raw_te, label):
    aa, bb = np.polyfit(raw_va, y[kva], 1)
    pe = np.exp(aa * raw_te + bb)
    print(f'{label}: overall {resolution(pe, Et[kte])["sigma_eff"]:.4f}')
    return pe
va_plain = infer(kva, False); te_plain = infer(kte, False)
pe_plain = calib_eval(va_plain.mean(0), te_plain.mean(0), '7-model ensemble (no TTA, reproduces nb34)')
va_tta = infer(kva, True); te_tta = infer(kte, True)
pe_tta = calib_eval(va_tta.mean(0), te_tta.mean(0), '7-model ensemble + D4 TTA')

7-model ensemble (no TTA, reproduces nb34): overall 0.0445


7-model ensemble + D4 TTA: overall 0.0440


## Part 2 - high-E specialist (predicted-energy gate at 15 GeV)

In [6]:
ens_tr = None
tr_raw = infer(ktr, False).mean(0)
aa, bb = np.polyfit(va_plain.mean(0), y[kva], 1)
pred_tr = np.exp(aa * tr_raw + bb); pred_va = np.exp(aa * va_plain.mean(0) + bb); pred_te = np.exp(aa * te_plain.mean(0) + bb)
str_ = np.asarray(ktr)[pred_tr > 15.0]; sva = np.asarray(kva)[pred_va > 15.0]; ste = np.asarray(kte)[pred_te > 15.0]
print(f'specialist domain: train {len(str_)}/{len(ktr)}, va {len(sva)}, te {len(ste)}')
EPOCHS = {'smoke': 2, 'full': 80}[MODE]; PATIENCE = {'smoke': 99, 'full': 12}[MODE]
SEEDS = {'smoke': [0], 'full': [0, 1]}[MODE]
def train_spec(seed):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    la1, lb1 = np.polyfit(np.log1p(0.5 * sumE[str_]), y[str_], 1)
    mdl = SubNet(IN_DIM, 5, la1, lb1).to(DEVICE)
    opt = torch.optim.AdamW(mdl.parameters(), lr=CFG['lr'], weight_decay=CFG['wd'])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    ck = CKPT / f'nb35_spec_s{seed}.pt'
    def batches(idx, bs, sh):
        idx = np.asarray(idx)
        if sh: idx = rng.permutation(idx)
        for j in range(0, len(idx), bs): yield torch.from_numpy(idx[j:j+bs]).to(DEVICE)
    def vloss():
        mdl.eval(); s = 0.0; k = 0
        with torch.no_grad():
            for b in batches(sva, 256, False):
                s += nn.functional.huber_loss(mdl(Xc[b], Mc[b], G5c[b], Ec[b]), Yc[b], delta=CFG['huber_delta']).item(); k += 1
        return s / max(k, 1)
    best = 1e9; bstate = None; wait = 0; ep0 = 0
    if ck.exists():
        st = torch.load(ck, map_location=DEVICE)
        mdl.load_state_dict(st['model']); opt.load_state_dict(st['opt']); sched.load_state_dict(st['sched'])
        best = st['best']; bstate = st['bstate']; wait = st['wait']; ep0 = st['ep'] + 1
        rng = np.random.default_rng(seed + 1000 * ep0)
    for ep in range(ep0, EPOCHS):
        mdl.train()
        for b in batches(str_, CFG['batch'], True):
            opt.zero_grad()
            nn.functional.huber_loss(mdl(Xc[b], Mc[b], G5c[b], Ec[b]), Yc[b], delta=CFG['huber_delta']).backward()
            opt.step()
        sched.step(); vv = vloss()
        if vv < best - 1e-4: best = vv; bstate = copy.deepcopy(mdl.state_dict()); wait = 0
        else: wait += 1
        torch.save(dict(model=mdl.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(),
                        best=best, bstate=bstate, wait=wait, ep=ep), ck)
        if wait >= PATIENCE: break
    mdl.load_state_dict(bstate); mdl.eval()
    def run(idx):
        out = []
        with torch.no_grad():
            for b in batches(idx, 256, False): out.append(mdl(Xc[b], Mc[b], G5c[b], Ec[b]).cpu().numpy().ravel())
        return np.concatenate(out)
    rv = run(sva)
    a2, b2 = np.polyfit(rv, y[sva], 1)
    return np.exp(a2 * run(ste) + b2), np.exp(a2 * rv + b2)
spec_pairs = [train_spec(s) for s in SEEDS]
spec_ens = np.stack([p[0] for p in spec_pairs]).mean(0)
spec_va = np.stack([p[1] for p in spec_pairs]).mean(0)
mask_hi = pred_te > 15.0; mask_hi_va = pred_va > 15.0
aa_t, bb_t = np.polyfit(va_tta.mean(0), y[kva], 1)
tta_va = np.exp(aa_t * va_tta.mean(0) + bb_t)
print('branch choice on VALIDATION (pred>15 GeV):')
VA = {'tta_only': tta_va[mask_hi_va], 'spec': spec_va, 'blend': 0.5 * (tta_va[mask_hi_va] + spec_va)}
for k, v in VA.items(): print(f'  {k:9s} val sigma_eff {resolution(v, Et[sva])["sigma_eff"]:.4f}')
va_choice = min(VA, key=lambda k: resolution(VA[k], Et[sva])['sigma_eff'])
print('chosen on val:', va_choice)

specialist domain: train 36806/50787, va 7972, te 7941


branch choice on VALIDATION (pred>15 GeV):
  tta_only  val sigma_eff 0.0380
  spec      val sigma_eff 0.0419
  blend     val sigma_eff 0.0395
chosen on val: tta_only


## Final combined table

In [7]:
pe_final = pe_tta.copy()
cands = {'tta_only': pe_tta[mask_hi], 'spec': spec_ens, 'blend': 0.5 * (pe_tta[mask_hi] + spec_ens)}
pe_final[mask_hi] = cands[va_choice]
print(f'high-E branch (chosen on val): {va_choice}')
te_e = Et[kte]
print(f'FINAL: overall {resolution(pe_final, te_e)["sigma_eff"]:.4f} (record 0.0444)')
print('targets:            0.06 / 0.045 / 0.035 / 0.032 / 0.030 / 0.030')
edges = np.quantile(te_e, np.linspace(0, 1, 7))
for i in range(6):
    hi = edges[i+1] + (1e-9 if i == 5 else 0)
    mm = (te_e >= edges[i]) & (te_e < hi)
    print(f'  E {edges[i]:6.1f}-{edges[i+1]:6.1f} GeV: {resolution(pe_final[mm], te_e[mm])["sigma_eff"]:.4f}')
np.save(OUT / 'nb35_final_pred.npy', pe_final)
pd.DataFrame([dict(branch=va_choice, sigma_eff=resolution(pe_final, te_e)['sigma_eff'])]).to_csv(OUT / 'nb35_final.csv', index=False)

high-E branch (chosen on val): tta_only
FINAL: overall 0.0440 (record 0.0444)
targets:            0.06 / 0.045 / 0.035 / 0.032 / 0.030 / 0.030
  E    2.2-  10.7 GeV: 0.0668
  E   10.7-  17.4 GeV: 0.0499
  E   17.4-  24.0 GeV: 0.0369
  E   24.0-  34.1 GeV: 0.0382
  E   34.1-  53.1 GeV: 0.0366
  E   53.1- 100.0 GeV: 0.0374
